# VibNet-1D — Preparação dos Dados Processados (`processed/*.npy`) no Google Colab

Este notebook baixa as bases brutas via **`vibdata`** e gera as janelas de **1 segundo** (`Detrend` + `SimpleSplit`) na estrutura exata esperada pelos experimentos do **VibNet-1D** (incluindo `colab_time_scratch_ablation.ipynb` e `run_time_scratch_ablation.py`):

```text
/content/drive/MyDrive/VibNet_Project/processed/
├── CWRU_12k/<Condição>/<Classe>/*.npy          (12.000 pontos por janela de 1s)
├── CWRU_48k/<Condição>/<Classe>/*.npy          (48.000 pontos por janela de 1s)
├── HUST/<Condição>/<Classe>/*.npy              (51.200 pontos por janela de 1s)
├── IMS/<Condição>/<Classe>/*.npy               (20.000 pontos por janela de 1s)
├── MFPT/<Condição>/<Classe>/*.npy              (48.828 pontos por janela de 1s)
├── UOC/<Condição>/<Classe>/*.npy               (2.048 pontos por janela)
├── UORED/<Condição>/<Classe>/*.npy             (42.000 pontos por janela de 1s)
├── Electric_Motor/<Condição>/<Classe>/*.npy    (50.000 pontos por janela de 1s)
├── HUST_Gearbox/<Condição>/<Classe>/*.npy      (25.600 pontos por janela de 1s)
├── Mechanical_Gear/<Condição>/<Classe>/*.npy   (5.000 pontos por janela de 1s)
├── PU/<Condição>/<Classe>/*.npy                (64.000 pontos por janela de 1s)
└── UOEMD/<Condição>/<Classe>/*.npy             (42.000 pontos por janela de 1s)
```

### Roteiro
1. **Parâmetros:** escolha quais datasets baixar/processar (`RAW_DATASETS_TO_PROCESS`) e os caminhos no Google Drive.
2. **Setup e instalação:** clona `VibNet-1D` e instala o fork customizado da `vibdata`.
3. **Montagem do Google Drive:** conecta ao Drive e verifica espaço livre e datasets já processados.
4. **Download e processamento 1D (reutilizando `src.data.make_dataset`):** gera as janelas `.npy` no SSD local do Colab (muito mais rápido) e sincroniza cada dataset concluído para `MyDrive/VibNet_Project/processed/`.
5. **Auditoria final:** confere número de condições, classes, janelas `.npy` e comprimento físico de 1s em cada dataset no Google Drive.

## 1. Parâmetros de Configuração

Por padrão, `RAW_DATASETS_TO_PROCESS` inclui todas as **11 bases brutas** (`CWRU`, `HUST`, `IMS`, `MFPT`, `UOC`, `UORED`, `Electric_Motor`, `HUST_Gearbox`, `Mechanical_Gear`, `PU`, `UOEMD`), gerando os 12 diretórios processados (pois `CWRU` divide-se automaticamente em `CWRU_12k` e `CWRU_48k`).

In [ ]:
# ---- Repositório ----
REPO_URL = "https://github.com/vfrocha/VibNet-1D.git"
BRANCH = "main"
REPO_DIR = "/content/VibNet-1D"

# ---- Google Drive ----
DRIVE_BASE = "/content/drive/MyDrive/VibNet_Project"
DRIVE_PROCESSED_DIR = f"{DRIVE_BASE}/processed"

# Onde baixar os arquivos brutos da vibdata:
# - Em "/content/vibnet_raw" (disco local da VM do Colab): não consome espaço do Drive com arquivos brutos grandes
# - Ou mude para f"{DRIVE_BASE}/raw_data" se quiser guardar os brutos no Google Drive também
RAW_DOWNLOAD_DIR = "/content/vibnet_raw"

# Diretório de staging local (SSD do Colab) para gravar milhares de .npy rapidamente antes de copiar ao Drive
LOCAL_STAGING_PROCESSED_DIR = "/content/vibnet_processed_staging"

# Datasets brutos da vibdata a baixar e processar:
# Nota: "CWRU" gera automaticamente "CWRU_12k", "CWRU_48k" (e "CWRU_12k_Severity" se habilitado)
RAW_DATASETS_TO_PROCESS = [
    "CWRU",
    "HUST",
    "IMS",
    "MFPT",
    "UOC",
    "UORED",
    "Electric_Motor",
    "HUST_Gearbox",
    "Mechanical_Gear",
    "PU",
    "UOEMD",
]

# Se False, pula automaticamente datasets cujo diretório final já exista e contenha arquivos .npy no Drive
OVERWRITE_EXISTING = False

# Se False, ignora a criação de CWRU_12k_Severity para economizar espaço e tempo no Drive
SAVE_CWRU_12K_SEVERITY = False

# Se True, remove os dados brutos de cada dataset após copiar as janelas .npy para o Drive (evita lotar o disco do Colab)
CLEAN_RAW_AFTER_DATASET = True

## 2. Clonagem do Repositório e Instalação da `vibdata`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Executando no Google Colab: {IN_COLAB}")

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print(f"Diretório atual: {Path.cwd()}")

# Instala dependências necessárias para download e descompactação da vibdata
!pip install -q gdown tqdm rarfile scipy numpy pandas scikit-learn
!pip install -q git+https://github.com/vfrocha/vibdata.git
!pip install -q -e .

# Utilitário unrar / 7z no Linux (usado por alguns datasets da vibdata)
if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "unrar", "p7zip-full"], check=False)

import vibdata.raw as raw_datasets
print("[OK] Biblioteca vibdata importada com sucesso.")

## 3. Montagem do Google Drive e Verificação de Espaço

In [ ]:
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

drive_proc_path = Path(DRIVE_PROCESSED_DIR)
raw_dl_path = Path(RAW_DOWNLOAD_DIR)
staging_proc_path = Path(LOCAL_STAGING_PROCESSED_DIR)

for p in (drive_proc_path, raw_dl_path, staging_proc_path):
    p.mkdir(parents=True, exist_ok=True)

os.environ["VIBNET_RAW_DIR"] = str(raw_dl_path)
os.environ["VIBNET_PROCESSED_DIR"] = str(drive_proc_path)

total_b, used_b, free_b = shutil.disk_usage("/content")
print(f"Espaço livre no disco local (/content): {free_b / (1024**3):.2f} GB")

if drive_proc_path.exists():
    existing_ds = sorted([d.name for d in drive_proc_path.iterdir() if d.is_dir()])
    print(f"Pasta no Drive: {drive_proc_path}")
    print(f"Datasets já existentes em processed/: {existing_ds}")

## 4. Download e Processamento de Janelas de 1 Segundo (`Detrend` + `SimpleSplit`)

Esta célula reutiliza exatamente os pipelines (`PIPELINES`), o extrator (`extract_signal`) e a função de nomenclatura (`get_names`) definidos em [`src/data/make_dataset.py`](file:///c:/Users/victor.rocha/Documents/VibNet-1D/src/data/make_dataset.py):
- Baixa cada base bruta via `vibdata.raw.<ds_name>_raw(RAW_DOWNLOAD_DIR, download=True)`.
- Grava as janelas `.npy` de 1 segundo primeiro no SSD local (`LOCAL_STAGING_PROCESSED_DIR`), evitando gargalo de I/O de milhares de arquivos pequenos no FUSE do Google Drive.
- Ao terminar cada dataset, copia a árvore `<dataset>/<condition>/<class>/*.npy` para `DRIVE_PROCESSED_DIR`.

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from tqdm.auto import tqdm
import vibdata.raw as raw_datasets
from src.data.make_dataset import PIPELINES, extract_signal, get_names


def harmonize_mfpt_sample_rate(signal, meta):
    # Harmoniza o MFPT para 48.828 Hz (reduz sinais de 97.656 Hz por fator 2)
    try:
        sample_rate = float(meta.get("sample_rate"))
    except (TypeError, ValueError):
        return signal
    target_rate = 48828.0
    if np.isclose(sample_rate, target_rate):
        return signal
    if np.isclose(sample_rate, 2 * target_rate):
        signal = np.asarray(signal).reshape(-1)
        return resample_poly(signal, up=1, down=2)
    return signal


def expected_target_names(raw_ds_name: str):
    if raw_ds_name == "CWRU":
        targets = ["CWRU_12k", "CWRU_48k"]
        if SAVE_CWRU_12K_SEVERITY:
            targets.append("CWRU_12k_Severity")
        return targets
    return [raw_ds_name]


def is_target_already_prepared(target_name: str) -> bool:
    target_dir = drive_proc_path / target_name
    if not target_dir.exists():
        return False
    return any(target_dir.rglob("*.npy"))


summary_counts = {}

for ds_name in RAW_DATASETS_TO_PROCESS:
    targets_for_ds = expected_target_names(ds_name)
    if not OVERWRITE_EXISTING and all(is_target_already_prepared(t) for t in targets_for_ds):
        print(f"\n[SKIP] {ds_name} -> {targets_for_ds} já existe(m) em {drive_proc_path}. Pulando.")
        continue

    print(f"\n{'=' * 70}")
    print(f" BAIXANDO E PROCESSANDO: {ds_name} -> {targets_for_ds}")
    print(f"{'=' * 70}")

    # Limpa staging local dos targets atuais
    for t_name in targets_for_ds:
        t_staging = staging_proc_path / t_name
        if t_staging.exists():
            shutil.rmtree(t_staging)

    try:
        raw_cls = getattr(raw_datasets, f"{ds_name}_raw")
        ds = raw_cls(str(raw_dl_path), download=True)
    except Exception as exc:
        print(f"[ERRO] Falha ao baixar/carregar {ds_name}: {exc}")
        continue

    saved_count = {}

    for i in tqdm(range(len(ds)), desc=f"Janelando {ds_name}"):
        try:
            item = ds[i]
            if not isinstance(item, dict):
                continue

            sig_array = extract_signal(item)
            if sig_array is None:
                continue

            meta = item["metainfo"]
            if isinstance(meta, pd.DataFrame):
                meta = meta.iloc[0]

            if ds_name == "MFPT":
                sig_array = harmonize_mfpt_sample_rate(sig_array, meta)

            if ds_name == "CWRU":
                sr = meta.get("sample_rate", 12000)
                if sr > 20000:
                    target_ds_names = ["CWRU_48k"]
                else:
                    target_ds_names = ["CWRU_12k"]
                    if SAVE_CWRU_12K_SEVERITY:
                        target_ds_names.append("CWRU_12k_Severity")
            else:
                target_ds_names = [ds_name]

            for target_ds_name in target_ds_names:
                current_transform = PIPELINES.get(target_ds_name)
                if not current_transform:
                    continue

                sample = {"signal": sig_array, "metainfo": pd.DataFrame([meta])}
                processed = current_transform(sample)
                windows = processed["signal"]

                if isinstance(windows, list) and len(windows) > 0:
                    cond, lbl = get_names(target_ds_name, meta)
                    final_dir = staging_proc_path / target_ds_name / cond / lbl
                    final_dir.mkdir(parents=True, exist_ok=True)

                    for idx, window in enumerate(windows):
                        if isinstance(window, np.ndarray):
                            fname = f"s{i:05d}_w{idx:02d}.npy"
                            np.save(final_dir / fname, window.astype(np.float32, copy=False))

                    saved_count[target_ds_name] = saved_count.get(target_ds_name, 0) + len(windows)

        except Exception as exc:
            print(f"\n[AVISO] Erro no registro {i} de {ds_name}: {exc}")
            continue

    print(f"--> Janelas geradas no staging local para {ds_name}: {saved_count}")

    # Copia cada target concluído do staging local para o Google Drive
    for target_ds_name, n_win in saved_count.items():
        src_t = staging_proc_path / target_ds_name
        dst_t = drive_proc_path / target_ds_name
        if not src_t.exists():
            continue
        print(f"  -> Copiando {target_ds_name} ({n_win} janelas) para o Google Drive: {dst_t} ...")
        if dst_t.exists() and OVERWRITE_EXISTING:
            shutil.rmtree(dst_t)
        shutil.copytree(src_t, dst_t, dirs_exist_ok=True)
        shutil.rmtree(src_t)
        summary_counts[target_ds_name] = n_win
        print(f"  -> [OK] {target_ds_name} sincronizado no Drive.")

    if CLEAN_RAW_AFTER_DATASET and str(raw_dl_path).startswith("/content/"):
        for child in raw_dl_path.iterdir():
            if child.is_dir():
                shutil.rmtree(child, ignore_errors=True)
            else:
                child.unlink(missing_ok=True)
        print(f"  -> Cache bruto local limpo ({raw_dl_path}).")

print("\n[CONCLUÍDO] Resumo de janelas salvas nesta sessão:", summary_counts)

## 5. Auditoria e Verificação da Pasta `processed/` no Google Drive

Confirma que todos os datasets em `MyDrive/VibNet_Project/processed/` possuem a hierarquia `<dataset>/<condition>/<class>/*.npy` e verifica o comprimento das janelas de 1 segundo.

In [ ]:
rows = []
for ds_dir in sorted([d for d in drive_proc_path.iterdir() if d.is_dir()]):
    cond_dirs = sorted([c for c in ds_dir.iterdir() if c.is_dir()])
    npy_files = list(ds_dir.rglob("*.npy"))
    classes = sorted({p.parent.name for p in npy_files})
    sample_len = int(np.load(npy_files[0]).shape[-1]) if npy_files else 0
    rows.append(
        {
            "Dataset": ds_dir.name,
            "Condições_LOCO": len(cond_dirs),
            "Classes_Únicas": len(classes),
            "Total_Janelas_npy": len(npy_files),
            "Pontos_por_Janela_1s": sample_len,
        }
    )

audit_df = pd.DataFrame(rows)
display(audit_df)
print("\n[OK] Dados prontos em:", drive_proc_path)
print("Agora você já pode executar o notebook notebooks/colab_time_scratch_ablation.ipynb!")